# Manual Labeling and Analysis

Creating the Gold Dataset.

In [9]:
import pandas as pd
import os

In [10]:
# Loading my 189k labeled dataset
DATA_DIR = os.path.join("..", "data")
os.makedirs(DATA_DIR, exist_ok=True)

df = pd.read_csv(os.path.join(DATA_DIR, 'comments_labeled.csv'))
df.head()

,comment_id,video_title,video_context,parent_context,text_preprocessed,category,sentiment,confidence,reasoning,api_key_used,tokens_in,tokens_out,parse_ok
0,UgzLPJdZudI_r4we6c94AaABAg,Arc Raiders Reveal Trailer | Game Awards 2021,Check out the world premiere reveal trailer fo...,NaN,Good choice of music,Game_Related,Pos,0.90,The primary topic of the comment is the game's...,…InyxBg,551,104,True
1,UgyAUoHvzWPWW5jVv994AaABAg,Arc Raiders Reveal Trailer | Game Awards 2021,Check out the world premiere reveal trailer fo...,NaN,this looks VERY fun,Game_Related,Pos,0.95,The primary topic of the comment is the game A...,…bceQ1A,551,113,True
2,UgzwutSd2BKZ6h-6dRF4AaABAg,Arc Raiders Reveal Trailer | Game Awards 2021,Check out the world premiere reveal trailer fo...,NaN,these graphics are sick!,Game_Related,Pos,0.95,The primary topic of the comment is the graphi...,…RLxqww,552,112,True
3,UgzlrEdJ-IeAuox6BLJ4AaABAg,Arc Raiders Reveal Trailer | Game Awards 2021,Check out the world premiere reveal trailer fo...,NaN,"Ok, Arc Raiders....I see you",Game_Related,Neu,0.80,"The primary topic is Arc Raiders, a game, whic...",…xT2YjA,555,115,True
4,UgwcSGsGzi8y4_ynAhh4AaABAg,Arc Raiders Reveal Trailer | Game Awards 2021,Check out the world premiere reveal trailer fo...,NaN,came here to shazam the song. The trailer is p...,Others,Pos,0.80,"The primary topic of the comment is the song, ...",…InyxBg,564,130,True


In [11]:
# Stratification
# We want roughly 250 comments per major aspect to ensure balanced validation
category_goals = {
    'AI_Voice_Related': 250,
    'Business_Model_Related': 250,
    'Game_Related': 250,
    'Others': 250}

gold_samples = []

for category, count in category_goals.items():
    # Filter for the aspect (using the column name from Llama output)
    subset = df[df['category'] == category]
    
    # If the subset is smaller than the goal, take everything available
    n_to_sample = min(len(subset), count)
    
    # Sample and add to our list
    gold_samples.append(subset.sample(n=n_to_sample, random_state=15))

# Combine and Shuffle
gold_df = pd.concat(gold_samples).sample(frac=1, random_state=15)

In [12]:
# Excluding Llama labels so we can label objectively
blind_cols = [
    'comment_id', 
    'text_preprocessed', 
    'video_title', 
    'video_context',
    'parent_context']

# Empty columns for me to fill
gold_df_blind = gold_df[blind_cols].copy()
gold_df_blind['gold_category'] = ""
gold_df_blind['gold_sentiment'] = ""

In [13]:
gold_df_blind.to_csv(os.path.join(DATA_DIR, 'gold_set_manual_labels.csv'), index=False)

print(f"Gold set of {len(gold_df_blind)} rows exported.")
print("Next Step: Open this in Excel/Google Sheets and fill in 'gold_category' and 'gold_sentiment'.")

Gold set of 1000 rows exported.
Next Step: Open this in Excel/Google Sheets and fill in 'gold_category' and 'gold_sentiment'.
